# Silver — Atendentes
Padroniza as seis colunas da base e remove apenas duplicatas completas.
Níveis válidos: 1 e 2. Desligamento vazio é permitido para ativos.
Dados inválidos interrompem a execução antes da gravação.

In [0]:
from config import BRONZE_SCHEMA, SILVER_SCHEMA
from utils import dq_check, dq_check_unique
from pyspark.sql import functions as F

atendentes_bronze = spark.table(f"{BRONZE_SCHEMA}.base_atendentes")

In [0]:
display(atendentes_bronze)

In [0]:
nivel = F.lower(F.trim(F.col('nivel_atendimento').cast('string')))
atendentes_silver = (
    atendentes_bronze.drop('ingestion_datetime').dropDuplicates()
    .withColumn('id_atendente', F.expr('try_cast(id_atendente AS INT)'))
    .withColumn('nome_atendente', F.initcap(F.trim(F.regexp_replace('nome_atendente', r'\s+', ' '))))
    .withColumn('nivel_atendimento',
                F.when(nivel.isin('1', 'nível 1', 'nivel 1', 'n1'), 1)
                 .when(nivel.isin('2', 'nível 2', 'nivel 2', 'n2'), 2))
    .withColumn('status_atendente', F.lower(F.trim('status_atendente')))
    .withColumn('_desligamento_original', F.trim('data_desligamento'))
)
for coluna in ['data_admissao', 'data_desligamento']:
    atendentes_silver = atendentes_silver.withColumn(
        coluna, F.try_to_date(F.trim(F.col(coluna)), 'yyyy-MM-dd')
    )

In [0]:
display(atendentes_silver)

In [0]:
_, passou = dq_check_unique('atendentes_silver', 'id único', atendentes_silver, ['id_atendente'])
if not passou:
    raise ValueError('Existem registros diferentes para o mesmo id_atendente.')

regras = {
    'id válido': F.col('id_atendente') > 0,
    'nome preenchido': F.length('nome_atendente') > 0,
    'nível válido': F.col('nivel_atendimento').isin(1, 2),
    'admissão válida': F.col('data_admissao').isNotNull(),

    'desligamento válido':    (F.col('data_desligamento').isNull() & (F.col('status_atendente') == 'ativo'))
    | (F.col('data_desligamento').isNotNull() & (F.col('status_atendente') == 'inativo'))
,
    'ordem das datas': (
        F.col('data_desligamento').isNull()
        | (F.col('data_desligamento') >= F.col('data_admissao'))
    ),
    'status coerente': (
        ((F.col('status_atendente') == 'ativo') & F.col('data_desligamento').isNull())
        | ((F.col('status_atendente') == 'inativo') & F.col('data_desligamento').isNotNull())
    ),
}
for nome, condicao in regras.items():
    _, passou = dq_check('atendentes_silver', nome, atendentes_silver, condicao)
    if not passou:
        raise ValueError(f'Falha na validação de atendentes: {nome}')

atendentes_silver = atendentes_silver.drop('_desligamento_original')

In [0]:
display(atendentes_silver)
#é valido deixar a data de desligamento com valores nulos desde que o status do atendente esteja ativo

In [0]:
atendentes_silver.write.format('delta').mode('overwrite').saveAsTable(f"{SILVER_SCHEMA}.base_atendentes")